
# UNet Fine-Tuning for Lake Segmentation — **RGB from Planet 8-Band**

This notebook trains a **UNet** on **RGB** channels extracted from **Planet 8-band** GeoTIFF tiles (size **512×512**).  
We use **ImageNet-pretrained** encoders (ResNet, EfficientNet, MobileNet, …) so transfer learning works as intended.

**What it does**
- Randomly sample **1,000** image/label pairs by matching filenames.
- Extract **RGB** from Planet 8-band tiles (defaults to **(6,4,2) = (Red, Green, Blue)** for SuperDove SR).
- Train/Val split (80/20).
- Train UNet across multiple **backbones** using **pretrained(ImageNet)** weights.
- **Loss:** BCE + Soft Dice (good for class imbalance).
- **Metrics:** IoU, Precision, Recall, F1, Accuracy on **train + val**, logged to CSV.
- Save **best-train-IoU** checkpoint per backbone.

> If your band ordering is different, **change `RGB_BANDS`** below.


## Dependencies

In [ ]:

# If needed, uncomment to install:
# %pip install -q torch torchvision timm segmentation-models-pytorch rasterio albumentations pandas scikit-image


## Configuration

In [10]:

from pathlib import Path
import random

# === REQUIRED: Paths ===
IMAGES_DIR = Path("D:/Thesis/glacial_lake_detection_thesis/Training/3 Preparing train data/rgb_outputs/subset_750/images")  # change to your imagery folder
LABELS_DIR = Path("D:/Thesis/glacial_lake_detection_thesis/Training/3 Preparing train data/rgb_outputs/subset_750/labels")  # change to your label folder (same filenames)

# === PlanetScope SuperDove SR default band mapping (1-indexed) ===
RGB_BANDS = (3, 2, 1)  # (R, G, B) 1-indexed

# === Outputs ===
OUTPUT_DIR = Path("./training_output_rgb/subset_750")
(OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)

# === Training config ===
NUM_SAMPLES = 750
RANDOM_SEED = 42
VAL_RATIO = 0.2
BATCH_SIZE = 2
NUM_WORKERS = 0
EPOCHS = 10
LR = 1e-3

BACKBONES = ['mit_b0', 'mit_b1', 'mit_b2', 'mit_b3', 'mit_b4', 'mit_b5']

random.seed(RANDOM_SEED)


## Discover Pairs & Sample

In [2]:

def list_pairs(images_dir: Path, labels_dir: Path):
    img_files = {p.name: p for p in images_dir.glob("*.tif")}
    lbl_files = {p.name: p for p in labels_dir.glob("*.tif")}
    common = sorted(set(img_files) & set(lbl_files))
    return [(img_files[n], lbl_files[n]) for n in common]

all_pairs = list_pairs(IMAGES_DIR, LABELS_DIR)
if not all_pairs:
    raise RuntimeError(f"No paired .tif files found under {IMAGES_DIR} & {LABELS_DIR}.")

print(f"Found {len(all_pairs)} pairs.")
random.shuffle(all_pairs)
sampled_pairs = all_pairs[:min(NUM_SAMPLES, len(all_pairs))]

split_idx = int(len(sampled_pairs) * (1 - VAL_RATIO))
train_pairs = sampled_pairs[:split_idx]
val_pairs = sampled_pairs[split_idx:]
print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")


Found 750 pairs.
Train: 600, Val: 150



## Dataset & Dataloaders (RGB extraction)
- Reads 8-band imagery and **selects RGB channels** via `RGB_BANDS` (1-indexed).
- Per-image **min–max** normalization to [0,1].
- Light flips for augmentation.


In [3]:

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import rasterio
import albumentations as A

def read_geotiff(path: Path):
    with rasterio.open(path) as src:
        arr = src.read()  # (C,H,W), 1-indexed bands in file
    return arr

def select_rgb(x: np.ndarray, rgb_bands=(6,4,2)):
    idx = [b-1 for b in rgb_bands]
    return x[idx, ...].astype(np.float32)

def minmax_per_band(x: np.ndarray, eps=1e-6):
    x = x.astype(np.float32)
    for i in range(x.shape[0]):
        vmin = np.min(x[i]); vmax = np.max(x[i])
        if vmax - vmin < eps:
            x[i] = 0.0
        else:
            x[i] = (x[i] - vmin) / (vmax - vmin + eps)
    return x

class LakeTilesRGB(Dataset):
    def __init__(self, pairs, rgb_bands=(6,4,2), aug=None, normalize=True):
        self.pairs = pairs
        self.rgb_bands = rgb_bands
        self.aug = aug
        self.normalize = normalize

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.pairs[idx]
        img8 = read_geotiff(img_path)          # (8,H,W)
        img = select_rgb(img8, self.rgb_bands) # (3,H,W)

        with rasterio.open(lbl_path) as src:
            lbl = src.read(1)                  # (H,W)
        lbl = (lbl > 0).astype(np.float32)

        if self.normalize:
            img = minmax_per_band(img)

        # Albumentations expects HWC
        img_hwc = np.transpose(img, (1,2,0))
        lbl_hwc = lbl[..., None]

        if self.aug:
            transformed = self.aug(image=img_hwc, mask=lbl_hwc)
            img_hwc = transformed["image"]
            lbl_hwc = transformed["mask"]

        # Back to CHW
        img = torch.from_numpy(np.transpose(img_hwc, (2,0,1))).float()
        lbl = torch.from_numpy(lbl_hwc[...,0]).float()
        return img, lbl

train_aug = A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5)])
val_aug   = A.Compose([])

train_ds = LakeTilesRGB(train_pairs, rgb_bands=RGB_BANDS, aug=None, normalize=True)
val_ds   = LakeTilesRGB(val_pairs,   rgb_bands=RGB_BANDS, aug=None,   normalize=True)

PIN_MEMORY = False
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          pin_memory=PIN_MEMORY)

len(train_ds), len(val_ds)


(600, 150)

## Loss & Metrics

In [4]:

import torch.nn.functional as F

def dice_loss_with_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    if probs.dim() == 4:
        probs = probs[:,0]
    intersection = (probs * targets).sum(dim=(1,2))
    union = probs.sum(dim=(1,2)) + targets.sum(dim=(1,2)) + eps
    dice = (2 * intersection + eps) / union
    return 1 - dice.mean()

def bce_dice_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets) + dice_loss_with_logits(logits, targets)

def compute_metrics_from_logits(logits, targets):
    with torch.no_grad():
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        t = targets.float()
        tp = (preds * t).sum(dim=(1,2))
        tn = ((1-preds) * (1-t)).sum(dim=(1,2))
        fp = (preds * (1-t)).sum(dim=(1,2))
        fn = ((1-preds) * t).sum(dim=(1,2))
        eps = 1e-7
        precision = (tp / (tp + fp + eps)).mean().item()
        recall    = (tp / (tp + fn + eps)).mean().item()
        f1        = (2*precision*recall / (precision + recall + eps))
        iou       = (tp / (tp + fp + fn + eps)).mean().item()
        accuracy  = ((tp + tn) / (tp + tn + fp + fn + eps)).mean().item()
        return dict(iou=iou, precision=precision, recall=recall, f1=f1, accuracy=accuracy)


## Model & Training (ImageNet-pretrained encoders, in_channels=3)

In [8]:

import segmentation_models_pytorch as smp
import torch
from torch import optim
import pandas as pd
from datetime import datetime
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def make_segformer_rgb(backbone: str, num_classes: int = 1, pretrained=True):
    encoder_weights = "imagenet" if pretrained else None
    model = smp.Unet(
        encoder_name=backbone,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=num_classes,
    )
    return model

def run_one_epoch(model, loader, optimizer=None, use_amp=True):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0; n_batches = 0
    agg = dict(iou=0.0, precision=0.0, recall=0.0, f1=0.0, accuracy=0.0)

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        if is_train: optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)[:,0]
            loss = bce_dice_loss(logits, lbls)

        if is_train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item(); n_batches += 1
        m = compute_metrics_from_logits(logits.detach(), lbls.detach())
        for k in agg: agg[k] += m[k]

    avg_loss = total_loss / max(1, n_batches)
    for k in agg: agg[k] /= max(1, n_batches)
    torch.cuda.empty_cache()
    return avg_loss, agg

# Determinism tweaks (optional)
#torch.backends.cudnn.benchmark = False
#torch.backends.cudnn.deterministic = True


Device: cuda


In [11]:
results_csv = OUTPUT_DIR / "logs" / "results.csv"
if not results_csv.exists():
    pd.DataFrame(columns=[
        "timestamp","backbone","epoch","split",
        "loss","iou","precision","recall","f1","accuracy"
    ]).to_csv(results_csv, index=False)

for backbone in BACKBONES:
    print(f"\n=== Training backbone: {backbone} ===")
    model = make_segformer_rgb(backbone, num_classes=1, pretrained=True).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    best_val_iou = -1.0
    best_path = OUTPUT_DIR / "models" / f"{backbone}_best_train_iou.pt"

    for epoch in range(1, EPOCHS+1):
        train_loss, train_metrics = run_one_epoch(model, train_loader, optimizer, use_amp=True)
        val_loss,   val_metrics   = run_one_epoch(model, val_loader, optimizer=None, use_amp=True)

        print(f"[{backbone}] Epoch {epoch}/{EPOCHS} — "
              f"train IoU={train_metrics['iou']:.4f}  val IoU={val_metrics['iou']:.4f}  "
              f"train Loss={train_loss:.4f}  val Loss={val_loss:.4f}")

        ts = datetime.utcnow().isoformat()
        pd.DataFrame([
            dict(timestamp=ts, backbone=backbone, epoch=epoch, split="train", loss=train_loss, **train_metrics),
            dict(timestamp=ts, backbone=backbone, epoch=epoch, split="val",   loss=val_loss,   **val_metrics),
        ]).to_csv(results_csv, mode="a", index=False, header=False)

        if val_metrics["iou"] > best_val_iou:
            best_val_iou = val_metrics["iou"]
            torch.save({
                "backbone": backbone,
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val_iou": best_val_iou,
            }, best_path)

print(f"Training complete. Logs: {results_csv}")
print(f"Models saved under: {OUTPUT_DIR / 'models'}")



=== Training backbone: mit_b0 ===


config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/14.3M [00:00<?, ?B/s]

C:\Users\gg\AppData\Local\Temp\ipykernel_23900\3096622034.py:27: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\gg\AppData\Local\Temp\ipykernel_23900\3096622034.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


...

config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/246M [00:00<?, ?B/s]

[mit_b4] Epoch 1/10 — train IoU=0.2597  val IoU=0.3910  train Loss=0.8234  val Loss=0.6145
[mit_b4] Epoch 2/10 — train IoU=0.3926  val IoU=0.4517  train Loss=0.6113  val Loss=0.5405


MemoryError: Unable to allocate 3.00 MiB for an array with shape (3, 512, 512) and data type float32